# 板内图案演示（不需要 HDMI 输入）

> **最省事：菜单 `Run` → `Run All Cells`，然后看采集卡。**
>
> 如果想一格一格跑，必须**从第一格开始按顺序**，因为后面的格要用第一格加载好的
> `ol` / `cfg` / `lat`。跳过第一格就会报 `name 'pd' is not defined`。

预期：采集卡里出现**彩条**（白 黄 青 绿 品红 红 蓝 黑）。
看不到画面时依次检查：采集卡 USB 插好、相机/设置里选的是 `USB Video`、
板子 HDMI OUT 接到采集卡输入。


## 第 1 格（必须先跑）：加载位流、配置输出时序

In [ ]:
from pynq import Overlay
from pynq.lib.video import VideoMode
import pattern_demo as pd

ol = Overlay("enhance_pattern.bit")
video = ol.video
video.hdmi_out.configure(VideoMode(1280, 720, 24, 60))
cfg = pd.find(ol, "gpio_enh_cfg")
lat = pd.find(ol, "gpio_enh_lat")
print("overlay loaded, hdmi_out = 1280x720@60")
print("capture card should now show colour bars")


## 第 2 格：把延迟读出来

`latency` 是内核流水线填充延迟（两个时间戳相减）。看到具体数字，说明像素确实在流动。


In [ ]:
if "cfg" not in globals():
    raise SystemExit("请先运行第一格（Overlay 加载那格），再运行这一格。"
                     "最省事的做法：菜单 Run -> Run All Cells。")
a, b = lat.mmio.read(0x00), lat.mmio.read(0x08)
n = (b - a) & 0xFFFFFFFF
print(f"in={a} out={b} -> latency {n} pixel clocks (~{n / 142.857:.3f} us at 142.857 MHz)")


## 第 3 格：换一个预设，画面立刻变

把 `PRESET` 改成下面任意一个再运行这一格：

| 预设 | 采集卡上应该看到 |
|---|---|
| `binarize` | 彩条变成纯黑纯白的硬边 |
| `invert` | 颜色反相并变灰 |
| `split` | **左边约 1/4 保持原样，右边变色**（最直观） |
| `night` | 整体变亮、暗部抬起 |
| `pass` | 和原图一致（对照组） |


In [ ]:
if "cfg" not in globals():
    raise SystemExit("请先运行第一格（Overlay 加载那格），再运行这一格。"
                     "最省事的做法：菜单 Run -> Run All Cells。")
PRESET = "split"          # 改成 binarize / invert / split / night / pass
lo, hi = pd.pack(pd.PRESETS[PRESET])
cfg.mmio.write(0x00, lo)
cfg.mmio.write(0x08, hi)
a, b = lat.mmio.read(0x00), lat.mmio.read(0x08)
n = (b - a) & 0xFFFFFFFF
print(f"preset {PRESET} applied; latency {n} pixel clocks (~{n / 142.857:.3f} us)")


## 第 4 格：自动轮播（录视频就用这一格）

每 `HOLD` 秒换一个预设，采集卡画面跟着变。录屏或手机拍 30 秒即可。
想停下就点工具栏的 ⏹。


In [ ]:
if "cfg" not in globals():
    raise SystemExit("请先运行第一格（Overlay 加载那格），再运行这一格。"
                     "最省事的做法：菜单 Run -> Run All Cells。")
import time

HOLD = 4
ROUNDS = 2
ORDER = ["pass", "clarity", "document", "night", "invert", "binarize", "split"]

for r in range(ROUNDS):
    for name in ORDER:
        lo, hi = pd.pack(pd.PRESETS[name])
        cfg.mmio.write(0x00, lo)
        cfg.mmio.write(0x08, hi)
        a, b = lat.mmio.read(0x00), lat.mmio.read(0x08)
        n = (b - a) & 0xFFFFFFFF
        print(f"[{time.strftime('%H:%M:%S')}] {name:<9} latency {n} clocks (~{n / 142.857:.3f} us)")
        time.sleep(HOLD)
print("done")
